In [1]:
import kagglehub
import json
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error


c:\Users\glebo\Documents\University\Machine Learning\project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open("../kaggle_token.json", "r") as f:
    kaggle_acc = json.load(f)

os.environ["KAGGLE_USERNAME"] = kaggle_acc["username"]
os.environ["KAGGLE_KEY"] = kaggle_acc["key"]

path = kagglehub.competition_download('uds-machine-learning-2026-regression')

print("Path to competition files:", path)

Path to competition files: C:\Users\glebo\.cache\kagglehub\competitions\uds-machine-learning-2026-regression


In [3]:
train = pd.read_csv(path+"/train.csv")

TARGET = "Rented Bike Count"

train = train.dropna()

X = train.drop(columns=[TARGET, "Date"])
y = train[TARGET]

train.head(2)


,Kaggle_ID,Date,Hour,Temperature,Humidity,Wind speed,Visibility,Dew point temperature,Solar Radiation,Rainfall,Snowfall,Seasons,Holiday,Functioning Day,Record_id,Rented Bike Count
0,0,15/05/2018,11,24.8,36,1.8,973,8.7,2.7,0.0,0.0,Sprng,No Holiday,Yes,5137441,943
1,1,7/9/2018,23,12.9,0,2.4,1688,50.9,0.0,2.4,1.4,Sproing,No Holiday,No,1465289,156


In [4]:
# =========================
# 2. COLUMN SPLIT
# =========================
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

# =========================
# 3. PREPROCESSOR
# =========================
numeric_transformer = Pipeline([("scaler", StandardScaler())])

categorical_transformer = Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore"))])

preprocessor = ColumnTransformer(
    [
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ]
)

# =========================
# 4. PIPELINE
# =========================
pipeline = Pipeline([("prep", preprocessor), ("model", Ridge())])

# =========================
# 5. GRID SEARCH
# =========================
param_grid = {"model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
    verbose=1,
)

grid.fit(X, y)

# =========================
# 6. RESULTS
# =========================
print("\nBest alpha:", grid.best_params_["model__alpha"])
print("Best CV RMSE:", -grid.best_score_)

# =========================
# 7. FINAL MODEL
# =========================
final_pipeline = grid.best_estimator_

final_pipeline.fit(X, y)
print("\nBest alpha:", grid.best_params_["model__alpha"])
print("Best CV MSLE:", -grid.best_score_)

Fitting 5 folds for each of 5 candidates, totalling 25 fits


C:\Users\glebo\AppData\Local\Temp\ipykernel_10936\2608303555.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()



Best alpha: 1.0
Best CV RMSE: 51.78161611177448

Best alpha: 1.0
Best CV MSLE: 51.78161611177448


# Submission

In [5]:
import pandas as pd
import numpy as np

# =========================
# 11. TEST PREDICTIONS
# =========================
test = pd.read_csv(path+"/test.csv")

# Drop non-feature columns safely
test_X = test.drop(columns=["Date"])

# Predict using trained pipeline
preds = final_pipeline.predict(test_X)

# If using MSLE, ensure no negative predictions
preds = np.clip(preds, 0, None)

# =========================
# 12. SUBMISSION
# =========================
submission = pd.read_csv(path+"/sample_submission.csv")

submission["Rented Bike Count"] = preds

submission.to_csv("my_first_submission.csv", index=False)

print("submission created successfully")
print(submission.head())

submission created successfully
   Kaggle_ID  Rented Bike Count
0          0           0.000000
1          1          42.549892
2          2          25.862254
3          3          13.702804
4          4           0.000000
